In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import itertools
import json
import re

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from src.analysis import analogy_pseudocausal
from src.analysis.state_space import StateSpaceAnalysisSpec, \
    prepare_state_trajectory, aggregate_state_trajectory, flatten_trajectory
from src.datasets.speech_equivalence import SpeechHiddenStateDataset


In [3]:
base_model = "w2v2_pc_8"

model_class = "ffff_32-pc-mAP1"
model_name = "word_broad_10frames_fixedlen25"

inflection_results_path = "inflection_results.parquet"
all_cross_instances_path = "all_cross_instances.parquet"
false_friends_path = "false_friends.csv"

all_cross_instances_path = "outputs/analogy_morph/inputs/librispeech-train-clean-100/w2v2_pc/all_cross_instances.parquet"
experiments_path = "outputs/analogy_morph/inputs/librispeech-train-clean-100/w2v2_pc/experiments.json"
state_space_specs_path = f"outputs/analogy_morph/inputs/librispeech-train-clean-100/w2v2_pc/state_space_spec.h5"

train_dataset = "librispeech-train-clean-100"
# hidden_states_path = f"outputs/hidden_states/{base_model}/{train_dataset}.h5"
hidden_states_path = f"/scratch/jgauthier/{base_model}_{train_dataset}.h5"

embeddings_path = f"outputs/model_embeddings/{train_dataset}/{base_model}/{model_class}/{model_name}/{train_dataset}.npy"

output_dir = f"."

pos_counts_path = "data/pos_counts.pkl"

seed = 42

metric = "cosine"

agg_fns = [
    ("mean_within_cut", "phoneme")
]

In [4]:
with open(experiments_path, "r") as f:
    experiments = json.load(f)

## Prepare model representations

In [5]:
if embeddings_path == "ID":
    model_representations = SpeechHiddenStateDataset.from_hdf5(hidden_states_path).states
else:
    with open(embeddings_path, "rb") as f:
        model_representations: np.ndarray = np.load(f)
state_space_spec = StateSpaceAnalysisSpec.from_hdf5(state_space_specs_path)
assert state_space_spec.is_compatible_with(model_representations)

/scratch/jgauthier/transformers3/lib/python3.11/site-packages/tables/attributeset.py:322: DataTypeWarning: Unsupported type for attribute 'labels_are_repr' in node '/'. Offending HDF5 class: 8
  value = self._g_getattr(self._v_node, name)


In [6]:
trajectory = prepare_state_trajectory(model_representations, state_space_spec)
trajectory = aggregate_state_trajectory(trajectory, state_space_spec, agg_fns[0])

  0%|          | 0/32046 [00:00<?, ?it/s]

Aggregating:   0%|          | 0/32046 [00:00<?, ?label/s]

In [7]:
agg, agg_src = flatten_trajectory(trajectory)

## Prepare metadata

In [8]:
cuts_df = state_space_spec.cuts.xs("phoneme", level="level").drop(columns=["onset_frame_idx", "offset_frame_idx"])
cuts_df["label_idx"] = cuts_df.index.get_level_values("label").map({l: i for i, l in enumerate(state_space_spec.labels)})
cuts_df["frame_idx"] = cuts_df.groupby(["label", "instance_idx"]).cumcount()
cuts_df = cuts_df.reset_index().set_index(["label", "instance_idx", "frame_idx"]).sort_index()

cut_phonemic_forms = cuts_df.groupby(["label", "instance_idx"]).description.agg(' '.join)

In [9]:
all_cross_instances = pd.read_parquet(all_cross_instances_path).astype({"inflected_instance_idx": int})

In [10]:
all_cross_instances

,base_phones,orig_post_divergence,inflected,inflected_instance_idx,inflected_idx,inflected_phones,post_divergence,inflection,base,base_idx,base_instance_idx,next_phoneme_idx,next_phoneme,base_inflection,counterfactual,next_phoneme_cohort,index
0,AA K Y AH P EY SH AH N,Z,occupational,0,16088,AA K Y AH P EY SH AH N AH L,AH L,NNS-AH,occupation,4463.0,0.0,9,AH,NNS,False,|AH|,NaN
1,AA K Y AH P EY SH AH N,Z,occupational,0,16088,AA K Y AH P EY SH AH N AH L,AH L,NNS-AH,occupation,4463.0,1.0,9,AH,NNS,False,|AH|,NaN
2,AA K Y AH P EY SH AH N,Z,occupational,0,16088,AA K Y AH P EY SH AH N AH L,AH L,NNS-AH,occupation,4463.0,2.0,9,AH,NNS,False,|AH|,NaN
3,AA K Y AH P EY SH AH N,Z,occupational,0,16088,AA K Y AH P EY SH AH N AH L,AH L,NNS-AH,occupation,4463.0,3.0,9,AH,NNS,False,|AH|,NaN
4,AA K Y AH P EY SH AH N,Z,occupational,0,16088,AA K Y AH P EY SH AH N AH L,AH L,NNS-AH,occupation,4463.0,4.0,9,AH,NNS,False,|AH|,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21234538,T,EY T,types,9,11299,T AY P S,AY P S,ate_bound_false,None,NaN,NaN,1,AY,None,False,|AA|AE|AH|AO|AW|AY|EH|ER|EY|IH|IY|OW|OY|R|UH|U...,NaN
21234539,T,EY T,types,10,11299,T AY P S,AY P S,ate_bound_false,None,NaN,NaN,1,AY,None,False,|AA|AE|AH|AO|AW|AY|EH|ER|EY|IH|IY|OW|OY|R|UH|U...,NaN
21234540,T,EY T,types,11,11299,T AY P S,AY P S,ate_bound_false,None,NaN,NaN,1,AY,None,False,|AA|AE|AH|AO|AW|AY|EH|ER|EY|IH|IY|OW|OY|R|UH|U...,NaN
21234541,T,EY T,types,12,11299,T AY P S,AY P S,ate_bound_false,None,NaN,NaN,1,AY,None,False,|AA|AE|AH|AO|AW|AY|EH|ER|EY|IH|IY|OW|OY|R|UH|U...,NaN


## Prepare prediction equivalences

In [11]:
# Collect prediction equivalence information. This means, for any given target class
# (e.g. the phoneme "S" at position 0), a marking of all tokens (units within words)
# which are a valid match.
# 
# For the evaluation `matches_next_phoneme`, this means only those tokens which are 
# in onset position and have the identity `S`. For the weaker evaluation `matches_next_phoneme_weak`,
# this means all tokens from any word that has the sound `S` in onset position.
# (Predicting any frame of such a word is equally good under the weak evaluation.)

from collections import defaultdict
infl_phones = set(tuple(inflected_phones.strip().split())
                        for inflected_phones in all_cross_instances.query("counterfactual == False")["inflected_phones"].unique())
all_prediction_equivalences = defaultdict(lambda: {
    # "matches_next_phoneme": set(),
    "matches_next_phoneme_weak": set(),
    # "matches_cohort": set()
})

In [12]:
# Map from label, instance idx to list of matching flat idxs
instance_to_flat_idxs = defaultdict(list)
for flat_idx, (label_idx, instance_idx, unit_idx) in enumerate(agg_src):
    instance_to_flat_idxs[state_space_spec.labels[label_idx], instance_idx].append(flat_idx)

In [13]:
for (next_unit, next_unit_idx), rows in tqdm(all_cross_instances.query("counterfactual == False").groupby(["next_phoneme", "next_phoneme_idx"])):
    match_frames = None
    for inflected_phones in infl_phones:
        if next_unit_idx < len(inflected_phones) and inflected_phones[next_unit_idx] == next_unit:
            if match_frames is None:
                match_frames = set(itertools.chain.from_iterable(
                    instance_to_flat_idxs[row.inflected, row.inflected_instance_idx] for row in rows.itertuples()
                ))

            key = (" ".join(inflected_phones[:next_unit_idx + 1]),)
            all_prediction_equivalences[key]["matches_next_phoneme_weak"] |= \
                match_frames
            # all_prediction_equivalences[" ".join(inflected_phones), next_unit_idx]["matches_next_phoneme"] |= \
            #     set(rows[rows.frame_idx == next_unit_idx].traj_flat_idx)

# for (cohort, next_unit_idx), rows in tqdm(all_cross_instances.head(1000).groupby(["base_phones", "next_phoneme_idx"])):
#     cohort_length = cohort.count(" ") + 1
#     match_frames = None
#     for inflected_phones in infl_phones:
#         if len(inflected_phones) > cohort_length and " ".join(inflected_phones[:cohort_length]) == cohort:
#             if match_frames is None:
#                 match_frames = set(itertools.chain.from_iterable(
#                     instance_to_flat_idxs[row.inflected, row.inflected_instance_idx] for row in rows.itertuples()
#                 ))

#             all_prediction_equivalences[" ".join(inflected_phones), next_unit_idx]["matches_cohort"] |= \
#                 match_frames

100%|██████████| 277/277 [00:22<00:00, 12.46it/s]


In [14]:
def study_equiv(label, next_unit_idx):
    key = (" ".join(label.split()[:next_unit_idx]),)
    sample = all_prediction_equivalences[key]["matches_next_phoneme_weak"]
    sample = np.random.choice(list(sample), size=min(5, len(sample)), replace=False)
    for flat_idx in sample:
        label_idx, instance_idx, unit_idx = agg_src[flat_idx]
        print(cut_phonemic_forms.loc[state_space_spec.labels[label_idx]].loc[instance_idx])

In [15]:
study_equiv("R IY Z AH N", 4)

B OW N AH P AA R T IH S T
K AA M AH N
AH NG K AH L
L AO T AH N Z
S T R AH K


In [16]:
all_prediction_equivalences = {
    key: {k: torch.tensor(list(vs)) for k, vs in equivs.items()}
    for key, equivs in all_prediction_equivalences.items()
}

## Behavioral tests

In [17]:
# experiment = "VBZNNS-src_VBZ_S-tgt_VBZ_Z-AH"
# config = experiments[experiment]
# config["equivalence_keys"] = ["inflected_phones"]
# config["prediction_equivalence_keys"] = ["to_inflected_phones"]
# ret = analogy_pseudocausal.run_experiment_equiv_level(
#     experiment, config, state_space_spec, all_cross_instances,
#     agg, agg_src,
#     cut_phonemic_forms=cut_phonemic_forms,
#     prediction_equivalences=all_prediction_equivalences,
#     num_samples=5,
#     device="cpu",
# )

In [18]:
# ret[["from_base_phones", "from", "from_post_divergence", "to_base_phones", "to", "to_inflected_phones", "matches_next_phoneme_weak_target_label", "matches_next_phoneme_weak_predicted_label", "matches_next_phoneme_weak_predicted_phones", "matches_next_phoneme_weak_target_rank", "matches_next_phoneme_weak_target_phones", "matches_next_phoneme_weak_average_precision"]]

In [19]:
# # post-hoc updates
# for config in experiments.values():
#     config["equivalence_keys"] = ["inflected_phones", "next_phoneme_idx"]
#     config["prediction_equivalence_keys"] = ["to_inflected_phones", "to_next_phoneme_idx"]

In [20]:
dev_expts = {k: v for _, (k, v) in zip(range(5), experiments.items())}

In [22]:
device = "cuda:2" if torch.cuda.is_available() else "cpu"
t_agg = torch.tensor(agg, device=device)
t_agg_src = torch.tensor(agg_src, device=device)

# pre-compute flat idx lookup
flat_idx_lookup = {(label_idx, instance_idx, phoneme_idx): flat_idx
                    for flat_idx, (label_idx, instance_idx, phoneme_idx) in enumerate(agg_src)}

experiment_results = pd.concat({
    experiment: analogy_pseudocausal.run_experiment_equiv_level(
        experiment, config,
        state_space_spec, all_cross_instances,
        t_agg, t_agg_src,
        flat_idx_lookup=flat_idx_lookup,
        cut_phonemic_forms=cut_phonemic_forms,
        prediction_equivalences=all_prediction_equivalences,
        num_samples=50,
        max_num_vector_samples=100,
        seed=seed,
        device=device)
    for experiment, config in tqdm(experiments.items(), unit="experiment")
}, names=["experiment"])
# experiment_results["correct"] = experiment_results.predicted_label == experiment_results.gt_label
experiment_results

  0%|          | 0/470 [00:00<?, ?experiment/s]

VBZNNS-src_VBZ_Z-tgt_VBZ_Z-AH


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 1/470 [00:04<35:13,  4.51s/experiment]

...

ate-src_False-tgt_False-N


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 470/470 [26:08<00:00,  3.34s/experiment]


matches_next_phoneme_weak_target_rank  \
experiment                                                                
VBZNNS-src_VBZ_Z-tgt_VBZ_Z-AH 0                                       4   
                              1                                     547   
                              2                                    2876   
                              3                                    2460   
                              4                                      85   
...                                                                 ...   
ate-src_False-tgt_False-N     45                                      1   
                              46                                      4   
                              47                                      1   
                              48                                      0   
                              49                                      8   

                                  matches_next_phoneme_weak_target_distance  \
experiment                                                                    
VBZNNS-src_VBZ_Z-tgt_VBZ_Z-AH 0                                    0.197780   
                              1                                    0.332257   
                              2                                    0.502819   
                              3                                    0.360834   
                              4                                    0.290739   
...                                                                     ...   
ate-src_False-tgt_False-N     45                                   0.327660   
                              46                                   0.160533   
                              47                                   0.251233   
                              48                                   0.130438   
                              49                                   0.192777   

                                  matches_next_phoneme_weak_target_label_idx  \
experiment                                                                     
VBZNNS-src_VBZ_Z-tgt_VBZ_Z-AH 0                                         3032   
                              1                                         5900   
                              2                                        12379   
                              3                                          672   
                              4                                        10654   
...                                                                      ...   
ate-src_False-tgt_False-N     45                                         117   
                              46                                         180   
                              47                                       16139   
                              48                                         180   
                              49                                       15944   

                                  matches_next_phoneme_weak_target_instance_idx  \
experiment                                                                        
VBZNNS-src_VBZ_Z-tgt_VBZ_Z-AH 0                                               2   
                              1                                              44   
                              2                                               2   
                              3                                              50   
                              4                                               2   
...                                                                         ...   
ate-src_False-tgt_False-N     45                                             48   
                              46                                             29   
                              47                                             14   
                              48                                            

### Save

In [23]:
output_dir = "analogy_pseudocausal_morph"
from pathlib import Path
Path(output_dir).mkdir(parents=True, exist_ok=True)

In [24]:
experiment_results.to_csv(f"{output_dir}/experiment_results.csv")

In [25]:
5

5